# funcoes_comuns

Funções compartilhadas pelos notebooks da camada **Silver** (`bronze-to-silver/`).

**Como usar:** no topo do seu notebook, em uma célula **sozinha**:

```
%run ./funcoes_comuns
```

**Regras**
- Só UMA PESSOA pode editar este notebook por vez!! Precisa de uma mudança? Peça na daily ou no grupo.
- Este notebook **só define funções**: nenhuma célula lê tabela ou executa ação, porque ele roda dentro de 5 notebooks.
- **Nunca** crie no seu notebook uma função com o mesmo nome de uma daqui: a sua sobrescreve a comum **sem aviso**.
- Problema que só a sua tabela tem → resolva no seu notebook. Se outra pessoa precisar da mesma solução, avise: ela é promovida para cá.
- Mecanismo fica aqui; regra de negócio fica no seu notebook (ex.: o `mapear_dominio` é comum, o dicionário de UFs é seu).
- **Nunca** use `.cast()` direto numa coluna da Bronze: use as funções `parse_*`. Com ANSI ligado (serverless), um valor sujo no `.cast()` derruba o Job.
- Ordem dos passos no notebook Silver: estrutura → tipos → domínios → duplicatas → consistência → órfãos (por último) → checagens → gravar.

**Seções**

| Seção | Funções |
|---|---|
| Configuração | imports e constantes |
| Leitura e gravação | `ler_bronze`, `gravar_silver` |
| Estrutura | `remover_cabecalho_repetido`, `aparar_textos` |
| Tipos | `parse_decimal`, `parse_inteiro`, `parse_data`, `parse_timestamp`, `somente_digitos` |
| Domínios e nulos | `padronizar_texto`, `mapear_dominio`, `valores_para_nulo` |
| Duplicatas, órfãos e flags | `deduplicar`, `separar_orfaos`, `criar_flag` |
| Checagens e log | `checar_unicidade`, `checar_sem_nulos`, `checar_dominio`, `checar_parse`, `reconciliar`, `registrar`, `gravar_log` |
| Exploração | `perfilar_coluna` |

## Configuração

Imports e constantes usados pelas funções abaixo. Funções e variáveis que começam com `_` são internas: não chame no seu notebook.

In [0]:
import unicodedata
from datetime import datetime

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Datas que significam "sem data" na origem (ex.: medidor sem retirada) e viram nulo no parse_data.
DATAS_SENTINELA = ("9999-12-31",)

# Textos que significam "sem valor" e viram nulo no valores_para_nulo (comparação sem caixa e sem espaços).
VALORES_NULOS = ("", "-", "--", "N/A", "NULL", "NONE")

# Destinos aceitos no registrar (log de qualidade).
DESTINOS_LOG = ("CORRIGIDO", "QUARENTENA", "FLAG", "REMOVIDO", "MANTIDO")

_ACENTOS = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑáàâãäéèêëíìîïóòôõöúùûüçñ"
_SEM_ACENTO = "AAAAAEEEEIIIIOOOOOUUUUCNaaaaaeeeeiiiiooooouuuucn"

_LOG_QUALIDADE = {}  # preenchido pelo registrar, gravado pelo gravar_log


def _col(c):
    """Aceita nome de coluna (str) ou Column."""
    return F.col(c) if isinstance(c, str) else c


def _colunas_negocio(df: DataFrame) -> list:
    """Colunas de negócio = todas menos as de rastreio (que começam com '_', ex.: _source_file)."""
    return [c for c in df.columns if not c.startswith("_")]


def _mostrar(df: DataFrame, n: int = 20):
    """display() no Databricks; show() fora dele."""
    try:
        display(df.limit(n))  # noqa: F821
    except NameError:
        df.show(n, truncate=False)

## Funções de leitura e gravação

Padronizam **onde** cada notebook lê e grava, para ninguém errar nome de catálogo, esquema ou prefixo `tb_`.
O catálogo **nunca** é escrito no código: leia do widget no seu notebook e passe adiante.

```
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")
```

### `ler_bronze(catalogo, tabela)`

Lê a tabela `<catalogo>.bronze.tb_<tabela>`. Você passa o nome **sem** o prefixo `tb_`.

```
uc = ler_bronze(CATALOGO, "unidades_consumidoras")
```

In [0]:
def ler_bronze(catalogo: str, tabela: str) -> DataFrame:
    """Lê <catalogo>.bronze.tb_<tabela>."""
    return spark.table(f"{catalogo}.bronze.tb_{tabela}")  # noqa: F821

### `gravar_silver(df, catalogo, tabela, quarentena=None)`

Grava a tabela Silver `<catalogo>.silver.tb_<tabela>` e a quarentena `tb_<tabela>_quarentena`, e devolve as contagens **lidas das tabelas gravadas**.

- Grava em `overwrite` com `overwriteSchema`: o notebook pode ser reexecutado quantas vezes for preciso.
- A quarentena é **sempre** criada, mesmo vazia, para as checagens e a Gold não quebrarem. Sem quarentena, passe `None`.
- A quarentena precisa ter a coluna `motivo_quarentena` (o `separar_orfaos` já cria). Ela é gravada na ordem: colunas de negócio → `motivo_quarentena` → rastreio (`_source_file`, `_ingestion_timestamp`).
- **Chame só depois das checagens e do `reconciliar`**, como manda o Padrão de Limpeza: se algo falhar, a Silver antiga fica intacta. As contagens devolvidas servem de confirmação do que foi gravado.

```
reconciliar("medidores", N_MD_BRONZE, md.count(), md_quarentena.count(), REMOVIDAS_MD)
n_md = gravar_silver(md, CATALOGO, "medidores", quarentena=md_quarentena)
```

In [0]:
def gravar_silver(df: DataFrame, catalogo: str, tabela: str, quarentena: DataFrame = None) -> dict:
    """Grava silver.tb_<tabela> e silver.tb_<tabela>_quarentena (sempre). Devolve {"silver": n, "quarentena": n}."""
    nome = f"{catalogo}.silver.tb_{tabela}"
    if quarentena is None:
        quarentena = df.limit(0).withColumn("motivo_quarentena", F.lit(None).cast("string"))
    if "motivo_quarentena" not in quarentena.columns:
        raise ValueError(f"[{tabela}] a quarentena precisa ter a coluna motivo_quarentena")
    # ordem da quarentena: negócio -> motivo_quarentena -> rastreio (_source_file, _ingestion_timestamp)
    negocio_q = [c for c in quarentena.columns if not c.startswith("_") and c != "motivo_quarentena"]
    rastreio_q = [c for c in quarentena.columns if c.startswith("_")]
    quarentena = quarentena.select(*negocio_q, "motivo_quarentena", *rastreio_q)
    for dados, destino in ((df, nome), (quarentena, f"{nome}_quarentena")):
        (dados.write.format("delta").mode("overwrite")
              .option("overwriteSchema", "true").saveAsTable(destino))
    contagem = {"silver": spark.table(nome).count(),  # noqa: F821
                "quarentena": spark.table(f"{nome}_quarentena").count()}  # noqa: F821
    print(f"[{tabela}] gravado: silver={contagem['silver']:,} | quarentena={contagem['quarentena']:,}")
    return contagem

## Funções de estrutura

Corrigem problemas na **forma** do arquivo, antes de olhar o conteúdo. São os primeiros passos de todo notebook Silver.

### `remover_cabecalho_repetido(df)`

Remove as linhas em que **cada** coluna de negócio contém o próprio nome da coluna. É o cabeçalho do CSV repetido no meio do arquivo (acontece em unidades e faturas).

Compara sem caixa e sem espaços nas pontas e ignora as colunas de rastreio (`_source_file`, `_ingestion_timestamp`). Uma linha com qualquer valor diferente do nome da coluna **não** é removida.

```
n_antes = df.count()
df = remover_cabecalho_repetido(df)
n_cabecalhos = n_antes - df.count()   # vai para o reconciliar e o registrar
```

In [0]:
def remover_cabecalho_repetido(df: DataFrame) -> DataFrame:
    """Remove linhas que repetem o cabeçalho (todas as colunas de negócio = nome da coluna)."""
    cond = None
    for c in _colunas_negocio(df):
        e = F.lower(F.trim(F.col(c))) == F.lit(c.lower())
        cond = e if cond is None else (cond & e)
    if cond is None:
        return df
    return df.filter(~F.coalesce(cond, F.lit(False)))

### `aparar_textos(df, colunas=None)`

Tira espaços, tabs e espaços não separáveis do **início e do fim** dos textos, e transforma texto vazio em **nulo**.
Ex.: `"Avenida dos Navegantes "` → `"Avenida dos Navegantes"`; `"   "` → nulo.

Sem `colunas`, aplica em todas as colunas de texto de negócio. Não mexe nos espaços do meio do texto (isso é com o `padronizar_texto`).

In [0]:
def aparar_textos(df: DataFrame, colunas: list = None) -> DataFrame:
    """Trim (espaço, tab, nbsp) nas pontas; vazio vira nulo."""
    if colunas is None:
        colunas = [c for c in _colunas_negocio(df) if isinstance(df.schema[c].dataType, T.StringType)]
    for c in colunas:
        v = F.regexp_replace(F.col(c), r"^[\s ]+|[\s ]+$", "")
        df = df.withColumn(c, F.when(v == "", None).otherwise(v))
    return df

## Funções de tipos

Convertem o texto da Bronze para o tipo certo. **Valor que não converte vira nulo** (nunca erro e nunca zero).
Por isso, toda conversão deve ser acompanhada do `checar_parse`, que falha se algum valor que existia virou nulo.

Convenção: R$ e kWh em `decimal(14,2)`; datas em `date`; data e hora em `timestamp`.

### `parse_decimal(col, precisao=14, escala=2)`

Converte valores monetários e de kWh para `decimal`. Aceita os dois formatos dos dados:
`"76.93"` e `"R$ 1.234,56"`, além de `"0,00"`, `"1234"`, negativos e o sinal `+` na frente (`"+12,50"`).

- Se o texto tem vírgula, a vírgula é o separador decimal e os pontos são de milhar.
- Valor com mais dígitos inteiros do que cabe (ex.: um ID que caiu na coluna de valor) ou com mais casas decimais do que a `escala` vira **nulo**, em vez de derrubar o Job. O `checar_parse` acusa esses casos.
- Por isso, `"1.234"` sem vírgula (mil ou 1,234?) vira nulo e é acusado, em vez de ser lido errado em silêncio. Nos nossos dados isso não acontece: o formato R$ sempre tem centavos.

```
df = df.withColumn("valor_total", parse_decimal("valor_total"))
```

In [0]:
def parse_decimal(col, precisao: int = 14, escala: int = 2):
    """Texto ('76.93', 'R$ 1.234,56', '+12,50') -> decimal(precisao, escala). Inválido ou grande demais vira nulo."""
    c = F.regexp_replace(_col(col), r"R\$|[\s\u00A0]", "")
    c = F.regexp_replace(c, r"^\+(?=\d)", "")                  # "+12,50" -> "12,50"
    c = F.when(c.contains(","), F.regexp_replace(F.regexp_replace(c, r"\.", ""), ",", ".")).otherwise(c)
    inteiros = precisao - escala
    decimais = rf"(\.\d{{1,{escala}}})?" if escala > 0 else ""
    padrao = rf"^-?\d{{1,{inteiros}}}{decimais}$"
    return F.when(c.rlike(padrao), c.cast(f"decimal({precisao},{escala})"))

### `parse_inteiro(col, tipo="int")`

Converte contagens e durações para inteiro (ex.: `duracao_min`, `sequencia`). Use `tipo="bigint"` para números grandes.

Aceita só dígitos (e sinal). `"74.0"`, `"74,5"` ou texto viram **nulo**, assim como números que não cabem no tipo. kWh **não** usa esta função: kWh é `decimal(14,2)`.

In [0]:
def parse_inteiro(col, tipo: str = "int"):
    """Texto -> int (até 9 dígitos) ou bigint (até 18). Aceita '+5'. Inválido vira nulo."""
    limites = {"int": 9, "bigint": 18}
    if tipo not in limites:
        raise ValueError(f"tipo deve ser um de {list(limites)}")
    c = F.regexp_replace(F.trim(_col(col)), r"^\+(?=\d)", "")      # "+5" -> "5"
    return F.when(c.rlike(rf"^-?\d{{1,{limites[tipo]}}}$"), c.cast(tipo))

### `parse_data(col, formatos=..., sentinelas=DATAS_SENTINELA)`

Converte texto para `date` testando vários formatos, na ordem. Formatos padrão:

| Exemplo | Formato | Leitura |
|---|---|---|
| `2023-03-15` | `yyyy-MM-dd` | ano-mês-dia |
| `15/03/2023` | `dd/MM/yyyy` | dia-mês-ano |
| `15.03.2023` | `dd.MM.yyyy` | dia-mês-ano |
| `03-15-2023` | `MM-dd-yyyy` | **mês**-dia-ano |

**Atenção:** nos dados da V-Energy, data com **hífen no fim** (`03-15-2023`) está em **mês-dia-ano**; com barra ou ponto, em dia-mês-ano. Isso foi validado contra o `mes_referencia` de faturas e leituras.

Datas-sentinela (`9999-12-31` = "sem data", ex.: medidor sem retirada) e datas inválidas viram **nulo**.

In [0]:
def parse_data(col, formatos=("yyyy-MM-dd", "dd/MM/yyyy", "dd.MM.yyyy", "MM-dd-yyyy"),
               sentinelas=DATAS_SENTINELA):
    """Texto -> date, testando os formatos na ordem. Sentinela e inválido viram nulo."""
    c = _col(col)
    d = F.coalesce(*[F.to_date(F.try_to_timestamp(c, F.lit(f))) for f in formatos])
    if not sentinelas:
        return d
    return F.when(d.cast("string").isin(list(sentinelas)), None).otherwise(d)

### `parse_timestamp(col, formatos=...)`

Igual ao `parse_data`, mas para data **com hora** (`timestamp`). Usado em ocorrências e atendimentos.

Formatos padrão: `2022-07-15 06:03:00`, `2022-07-15T06:03:00`, `2022-07-15 06:03`, `15/07/2022 06:03:00` e `15/07/2022 06:03`. Inválido vira **nulo**.

In [0]:
def parse_timestamp(col, formatos=("yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd HH:mm",
                                   "dd/MM/yyyy HH:mm:ss", "dd/MM/yyyy HH:mm")):
    """Texto -> timestamp, testando os formatos na ordem. Inválido vira nulo."""
    c = _col(col)
    return F.coalesce(*[F.try_to_timestamp(c, F.lit(f)) for f in formatos])

### `somente_digitos(col)`

Mantém só os dígitos. Tira a máscara de CPF/CNPJ e CEP: `"439.810.983-82"` → `"43981098382"`.
O resultado continua **texto** (zeros à esquerda importam). Se não sobrar nenhum dígito, vira nulo.

In [0]:
def somente_digitos(col):
    """Remove tudo que não é dígito. Sem dígitos vira nulo."""
    c = F.regexp_replace(_col(col), r"\D", "")
    return F.when(c == "", None).otherwise(c)

## Funções de domínios e nulos

Padronizam os valores de colunas categóricas (UF, classe, tipo, status...) e os textos que significam "sem valor".
Convenção: domínios em **MAIÚSCULAS e sem acento**.

### `padronizar_texto(col, remover_acentos=True)`

Deixa o texto em maiúsculas, junta espaços repetidos em um só e tira os espaços das pontas. Texto vazio vira nulo.

- Com `remover_acentos=True` (padrão), tira os acentos: `"Poder  Público"` → `"PODER PUBLICO"`. Use em **domínios**.
- Com `remover_acentos=False`, mantém os acentos: `"José da Silva"` → `"JOSÉ DA SILVA"`. Use em **nomes e endereços**, que aparecem na tela.

In [0]:
def padronizar_texto(col, remover_acentos: bool = True):
    """Maiúsculas, espaços únicos, sem espaços nas pontas; opcionalmente sem acento. Vazio vira nulo."""
    c = _col(col)
    if remover_acentos:
        c = F.translate(c, _ACENTOS, _SEM_ACENTO)
    c = F.trim(F.regexp_replace(F.upper(c), r"[\s ]+", " "))
    return F.when(c == "", None).otherwise(c)


def _padronizar_py(s):
    """Mesma padronização do padronizar_texto, em Python (usada nas chaves do mapear_dominio)."""
    if s is None:
        return None
    s = "".join(ch for ch in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(ch))
    s = " ".join(s.upper().split())
    return s or None

### `mapear_dominio(col, mapa, padrao=None)`

Padroniza o valor (com o `padronizar_texto`) e troca pelo destino do `mapa`.

- As **chaves do mapa também são padronizadas**: pode escrever `"Pernambuco"`, `"Poder Público"`, com ou sem acento.
- Valor que não está no mapa fica com o texto padronizado (ou com `padrao`, se você passar um).
- **Sempre** rode o `checar_dominio` depois: um valor novo fora do mapa passa adiante sem aviso.

O mapa é **regra de negócio** e fica no seu notebook:

```
MAPA_UF = {"PE": "PE", "Pernambuco": "PE", "P.E.": "PE", "pe": "PE"}
df = df.withColumn("uf", mapear_dominio("uf", MAPA_UF))
checar_dominio(df, "uf", ["PE"], "unidades")
```

In [0]:
def mapear_dominio(col, mapa: dict, padrao=None):
    """Padroniza o valor e troca pelos destinos do mapa (chaves também padronizadas)."""
    normalizado = {}
    for origem, destino in mapa.items():
        k = _padronizar_py(origem)
        if k in normalizado and normalizado[k] != destino:
            raise ValueError(f"chaves {origem!r} e outra viram {k!r} com destinos diferentes")
        normalizado[k] = destino
    c = padronizar_texto(col)
    expr = None
    for origem, destino in normalizado.items():
        cond = c == F.lit(origem)
        expr = F.when(cond, F.lit(destino)) if expr is None else expr.when(cond, F.lit(destino))
    resto = c if padrao is None else F.lit(padrao)
    return resto if expr is None else expr.otherwise(resto)

### `valores_para_nulo(col, valores=VALORES_NULOS)`

Transforma em **nulo** os textos que significam "sem valor": `""`, `"-"`, `"--"`, `"N/A"`, `"NULL"`, `"NONE"` (sem caixa e sem espaços nas pontas). Os demais valores ficam **exatamente** como estavam.

Rode **antes** do `parse_*` e do `checar_parse`, senão o `"-"` conta como falha de conversão.

**Sentinelas como `"-1"` e `"9999-12-31"` não estão no padrão de propósito**: `-1` pode ser um valor real (um consumo de −1 kWh é arredondamento). Onde eles significam "sem valor" (ex.: `leitura_anterior`, `cep`), passe explicitamente:

```
df = df.withColumn("leitura_anterior", valores_para_nulo("leitura_anterior", VALORES_NULOS + ("-1",)))
```

Se o seu caso pede um rótulo em vez de nulo, complete depois no seu notebook:

```
df = df.withColumn("causa", F.coalesce(valores_para_nulo("causa"), F.lit("NAO INFORMADA")))
```

In [0]:
def valores_para_nulo(col, valores=VALORES_NULOS):
    """Textos de 'sem valor' viram nulo; o resto fica igual."""
    c = _col(col)
    alvo = [v.strip().upper() for v in valores]
    return F.when(F.upper(F.trim(c)).isin(alvo), None).otherwise(c)

## Funções de duplicatas, órfãos e flags

### `deduplicar(df, chave=None, ordem=None)`

Tem **dois usos**, que são decisões diferentes:

**1. `deduplicar(df)`: remove linhas idênticas.** Compara só as colunas de negócio e ignora as de rastreio. Assim, a mesma leitura que aparece em dois CSVs (com `_source_file` diferente) é removida. Entre as cópias, fica a de menor `_source_file` (resultado sempre igual entre execuções). É seguro: é a mesma linha repetida.

**2. `deduplicar(df, chave, ordem)`: resolve conflitos.** Quando a mesma chave aparece com dados **diferentes**, mantém uma linha por chave: a primeira segundo a `ordem`. A `ordem` é **obrigatória**: quem resolve um conflito precisa dizer qual linha vence e registrar essa regra no `registrar`. Se a `ordem` empatar, as colunas de negócio desempatam, para o resultado não mudar entre execuções.

Passar `chave` sem `ordem` dá **erro de propósito**: sem ordem, o Spark escolheria uma linha qualquer e esconderia o conflito.

Na maioria dos casos, o certo é: padronizar → `deduplicar(df)` → `checar_unicidade(df, chave)`. Se o `checar_unicidade` falhar, há um conflito real, que precisa ser entendido antes de escolher uma regra.

```
df = deduplicar(df)                                                  # uso 1
df = deduplicar(df, ["id_uc"], [F.col("data_ligacao").desc()])      # uso 2
```

In [0]:
def deduplicar(df: DataFrame, chave: list = None, ordem: list = None) -> DataFrame:
    """Uso 1 (sem chave): remove linhas idênticas nas colunas de negócio.
    Uso 2 (chave + ordem): mantém a 1ª linha por chave segundo a ordem. Chave sem ordem é erro."""
    if chave is None and ordem is not None:
        raise ValueError("ordem só faz sentido com chave")
    if chave is not None and not ordem:
        raise ValueError(
            "deduplicar com chave exige ordem (qual linha vence?). "
            "Para remover só linhas idênticas, chame deduplicar(df) e depois checar_unicidade(df, chave).")
    negocio = _colunas_negocio(df)
    rastreio = [c for c in df.columns if c.startswith("_")]

    # 1) linhas idênticas nas colunas de negócio
    if rastreio:
        w = Window.partitionBy(*negocio).orderBy(*[F.col(c).asc_nulls_last() for c in rastreio])
        df = df.withColumn("__rn", F.row_number().over(w)).filter("__rn = 1").drop("__rn")
    else:
        df = df.dropDuplicates()
    if chave is None:
        return df

    # 2) uma linha por chave, pela ordem (colunas de negócio desempatam)
    desempate = [c for c in negocio if c not in chave and not isinstance(df.schema[c].dataType, T.MapType)]
    w = Window.partitionBy(*chave).orderBy(*ordem, *desempate)
    return df.withColumn("__rn", F.row_number().over(w)).filter("__rn = 1").drop("__rn")

### `separar_orfaos(df, referencia, col_df, col_ref=None, motivo=None)`

Integridade referencial: separa os registros cujo `col_df` **não existe** em `referencia.col_ref`. Devolve `(validos, quarentena)`, e a quarentena ganha a coluna `motivo_quarentena`.

- É o **último** passo de transformação: rode depois de padronizar e deduplicar os dois lados.
- Os dois lados precisam estar padronizados igual (`"UC123 "` ≠ `"UC123"`). Compare a quantidade de órfãos com a esperada. Se vier muito diferente, o problema é a chave, não o dado.
- Registro com `col_df` **nulo** também vai para a quarentena.

```
leituras, lt_orfas = separar_orfaos(leituras, medidores, "id_medidor",
                                    motivo="id_medidor inexistente em medidores")
```

In [0]:
def separar_orfaos(df: DataFrame, referencia: DataFrame, col_df: str, col_ref: str = None, motivo: str = None):
    """Separa registros cujo col_df não existe em referencia.col_ref. Retorna (validos, quarentena)."""
    col_ref = col_ref or col_df
    ref = referencia.select(F.col(col_ref).alias("__ref__")).distinct()
    cond = df[col_df] == ref["__ref__"]
    validos = df.join(ref, cond, "left_semi")
    orfaos = (df.join(ref, cond, "left_anti")
                .withColumn("motivo_quarentena", F.lit(motivo or f"{col_df} inexistente em {col_ref}")))
    return validos, orfaos

### `criar_flag(df, nome, condicao)`

Cria uma flag booleana. Garante o padrão do time:

- o nome precisa começar com `flag_`;
- o resultado é sempre `true` ou `false`, **nunca nulo**: quando a condição é nula (ex.: consumo nulo), a flag fica `false`.

```
df = criar_flag(df, "flag_estimada", F.col("tipo_leitura") == "ESTIMADA")
```

In [0]:
def criar_flag(df: DataFrame, nome: str, condicao) -> DataFrame:
    """Cria coluna boolean flag_*; condição nula vira false."""
    if not nome.startswith("flag_"):
        raise ValueError(f"o nome da flag precisa começar com 'flag_': {nome!r}")
    return df.withColumn(nome, F.coalesce(condicao.cast("boolean"), F.lit(False)))

## Funções de checagens e log

As checagens **param o notebook** (e o Job) com uma mensagem clara quando algo não fecha, para dado errado nunca chegar à Gold.
O log registra cada problema tratado (problema → regra → linhas afetadas) na tabela `silver.tb_log_qualidade`, que vira a planilha de problemas e alimenta o dicionário de dados.

### `checar_unicidade(df, chave, nome="")`

Falha se alguma combinação da `chave` aparece mais de uma vez, e mostra até 5 exemplos.

```
checar_unicidade(df, ["id_uc"], "unidades")
```

In [0]:
def checar_unicidade(df: DataFrame, chave: list, nome: str = ""):
    """Falha se houver chave repetida."""
    rep = df.groupBy(*chave).count().filter("count > 1")
    exemplos = rep.limit(5).collect()
    if exemplos:
        n = rep.count()
        raise AssertionError(f"[{nome}] {n:,} chaves repetidas em {chave}. Ex.: {[tuple(r) for r in exemplos]}")
    print(f"[{nome}] ok: {chave} é única")

### `checar_sem_nulos(df, colunas, nome="")`

Falha se alguma das `colunas` tem nulo. Confere todas numa única leitura.

```
checar_sem_nulos(df, ["id_uc", "classe", "municipio"], "unidades")
```

In [0]:
def checar_sem_nulos(df: DataFrame, colunas: list, nome: str = ""):
    """Falha se alguma coluna tiver nulo."""
    r = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas]).first().asDict()
    com_nulo = {c: n for c, n in r.items() if n}
    if com_nulo:
        raise AssertionError(f"[{nome}] nulos encontrados: {com_nulo}")
    print(f"[{nome}] ok: sem nulos em {colunas}")

### `checar_dominio(df, coluna, permitidos, nome="")`

Falha se a `coluna` tem algum valor fora da lista `permitidos` e mostra até 10 deles. Nulo não conta como fora do domínio (use o `checar_sem_nulos` se a coluna for obrigatória).

```
checar_dominio(df, "classe", ["RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "RURAL", "PODER PUBLICO"], "unidades")
```

In [0]:
def checar_dominio(df: DataFrame, coluna: str, permitidos: list, nome: str = ""):
    """Falha se houver valor (não nulo) fora de permitidos."""
    fora = (df.filter(F.col(coluna).isNotNull() & ~F.col(coluna).isin(list(permitidos)))
              .select(coluna).distinct().limit(10).collect())
    if fora:
        raise AssertionError(f"[{nome}] valores fora do domínio em {coluna}: {[r[0] for r in fora]}")
    print(f"[{nome}] ok: {coluna} dentro do domínio")

### `checar_parse(df, colunas, funcao, nome="", **parametros)`

Garante que uma conversão **não perdeu nenhum valor**. Para cada coluna, conta as linhas em que o texto original existia, mas a `funcao` devolveu nulo.
Se houver alguma, mostra até 10 dessas linhas **com as colunas vizinhas** (para ver se é formato novo ou valor deslocado) e falha.

- Rode **antes** de converter, com as colunas ainda em texto. Confere todas as colunas numa única leitura.
- Rode **depois** do `valores_para_nulo`, se a coluna tiver `"-"` e afins.
- Valor que é nulo por regra de negócio, como a sentinela `9999-12-31`, não conta como falha.
- Não há tolerância: se aparecer lixo legítimo, trate explicitamente no seu notebook (e registre).

```
VALORES = ["valor_energia", "tributos", "valor_total"]
checar_parse(df, VALORES, parse_decimal, "faturas")
df = df.withColumns({c: parse_decimal(c) for c in VALORES})
```

In [0]:
def checar_parse(df: DataFrame, colunas: list, funcao, nome: str = "", **parametros):
    """Falha se funcao(col) for nula onde o texto original existia (sentinelas não contam)."""
    sentinelas = list(parametros.get("sentinelas", DATAS_SENTINELA)) if funcao is parse_data else []
    falhas = {}
    for c in colunas:
        original = F.trim(F.col(c))
        perdeu = original.isNotNull() & (original != "") & funcao(c, **parametros).isNull()
        if sentinelas:
            perdeu = perdeu & ~original.isin(sentinelas)
        falhas[c] = perdeu
    r = df.agg(*[F.sum(e.cast("int")).alias(c) for c, e in falhas.items()]).first().asDict()
    com_falha = {c: n for c, n in r.items() if n}
    if com_falha:
        alguma = None
        for c in com_falha:
            alguma = falhas[c] if alguma is None else (alguma | falhas[c])
        print(f"[{nome}] exemplos de valores que não converteram com {funcao.__name__}:")
        _mostrar(df.filter(alguma), 10)
        raise AssertionError(f"[{nome}] {funcao.__name__} perdeu valores: {com_falha}")
    print(f"[{nome}] ok: {funcao.__name__} converteu 100% de {colunas}")

### `reconciliar(nome, bronze, silver, quarentena=0, removidas=0)`

Confere que nenhuma linha sumiu em silêncio: **bronze = silver + quarentena + removidas**. Imprime as contagens e falha se não fechar.

**Importante:** `removidas` tem que ser o que **cada passo mediu** (cabeçalhos, duplicatas...), não uma conta. Se você calcular `removidas = bronze - silver - quarentena`, a reconciliação fecha sempre e não testa nada.
Por isso, passe um dicionário com o que cada passo removeu.

Rode **antes** do `gravar_silver`: se a conta não fechar, o notebook para e a Silver antiga fica intacta.

```
reconciliar("unidades_consumidoras", N_UC_BRONZE, uc.count(), uc_quarentena.count(),
            removidas={"cabecalho": n_cab, "duplicatas": n_dup})
n_uc = gravar_silver(uc, CATALOGO, "unidades_consumidoras", quarentena=uc_quarentena)
```

In [0]:
def reconciliar(nome: str, bronze: int, silver: int, quarentena: int = 0, removidas=0):
    """Falha se bronze != silver + quarentena + removidas. removidas: int ou {passo: n}."""
    detalhe = removidas if isinstance(removidas, dict) else {"removidas": removidas}
    total_removidas = sum(detalhe.values())
    print(f"[{nome}] bronze={bronze:,} | silver={silver:,} | quarentena={quarentena:,} | "
          f"removidas={total_removidas:,} {detalhe if isinstance(removidas, dict) else ''}")
    diferenca = bronze - (silver + quarentena + total_removidas)
    if diferenca != 0:
        raise AssertionError(f"[{nome}] reconciliação não fecha: diferença de {diferenca:,} linhas")
    print(f"[{nome}] ok: reconciliação fecha")

### `registrar(tabela, problema, regra, destino, linhas)`

Anota um problema tratado no log de qualidade (em memória). O `destino` diz o que foi feito:

| Destino | Quando |
|---|---|
| `CORRIGIDO` | o valor foi corrigido na própria linha (ex.: UF padronizada) |
| `QUARENTENA` | a linha foi para a tabela de quarentena |
| `FLAG` | a linha ficou e ganhou uma flag |
| `REMOVIDO` | a linha foi removida (cabeçalho, duplicata idêntica) |
| `MANTIDO` | o problema é conhecido e o dado fica como está (ex.: CEP nulo) |

Registrar de novo o mesmo `(tabela, problema)` substitui o anterior, então reexecutar uma célula não duplica o log.
Nada vai para o banco até o `gravar_log`.

```
registrar("medidores", "duplicatas idênticas", "deduplicar(df)", "REMOVIDO", n_dup)
```

In [0]:
def registrar(tabela: str, problema: str, regra: str, destino: str, linhas: int):
    """Anota (em memória) um problema tratado. Gravado no banco pelo gravar_log."""
    destino = destino.upper()
    if destino not in DESTINOS_LOG:
        raise ValueError(f"destino deve ser um de {DESTINOS_LOG}")
    _LOG_QUALIDADE[(tabela, problema)] = (tabela, problema, regra, destino, int(linhas))
    print(f"[{tabela}] registrado: {problema} → {destino} ({int(linhas):,} linhas)")

### `gravar_log(catalogo, notebook)`

Grava o log de qualidade em `<catalogo>.silver.tb_log_qualidade`. Chame **uma vez, no fim** do notebook.

Substitui **só as linhas do seu notebook** (`replaceWhere notebook = ...`). Isso importa porque, no Job, 4 notebooks Silver rodam **em paralelo** e gravam na mesma tabela: um `overwrite` comum apagaria o log dos outros.
A tabela é particionada por `notebook` e criada se não existir. No Job, o `silver_cadastro` roda antes dos outros e a cria.

- O `registrado_em` usa o relógio do Spark (`current_timestamp`), o mesmo da Bronze.
- **Log vazio também é gravado:** se o notebook não registrou nenhum problema, as linhas antigas dele são apagadas. Assim o log nunca mostra problemas que já não existem.

```
gravar_log(CATALOGO, "silver_cadastro")
```

In [0]:
def gravar_log(catalogo: str, notebook: str):
    """Grava o log em silver.tb_log_qualidade, substituindo só as linhas deste notebook.
    Log vazio também é gravado: apaga as linhas antigas do notebook."""
    nome = f"{catalogo}.silver.tb_log_qualidade"
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {nome} (
        notebook STRING, tabela STRING, problema STRING, regra STRING,
        destino STRING, linhas BIGINT, registrado_em TIMESTAMP)
        USING DELTA PARTITIONED BY (notebook)""")  # noqa: F821
    linhas = [(notebook, *v[:4], v[4]) for v in _LOG_QUALIDADE.values()]
    esquema = ("notebook string, tabela string, problema string, regra string, "
               "destino string, linhas bigint")
    df = (spark.createDataFrame(linhas, esquema)  # noqa: F821
               .withColumn("registrado_em", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite")
       .option("replaceWhere", f"notebook = '{notebook}'").saveAsTable(nome))
    print(f"[{notebook}] log de qualidade gravado: {len(linhas)} problemas")

## Funções de exploração

**Não chame nos notebooks do pipeline.** Servem para investigar os dados antes de escrever as regras, em um notebook de rascunho.
Só exibem resultados e não devolvem nada, para ninguém encadear o resultado numa transformação por engano.

### `perfilar_coluna(df, coluna, n=10)`

Mostra um raio-x de uma coluna para caçar anomalias:

- **Texto:** total, nulos, distintos, menor e maior tamanho, quantos têm espaço nas pontas, quantos têm caractere fora do padrão e quantos parecem **encoding quebrado** (`"JordÃ£o"`). Depois, os `n` valores mais frequentes, os `n` primeiros e últimos em ordem alfabética (strings corrompidas costumam ir para as pontas) e exemplos com caracteres estranhos.
- **Número ou data:** resumo estatístico (mínimo, quartis, máximo, média, desvio) e os `n` menores e maiores valores.
- **Lista (array):** tamanho mínimo e máximo, listas vazias, listas nulas, listas com item nulo e com itens repetidos, e exemplos das maiores e menores.

```
perfilar_coluna(ler_bronze(CATALOGO, "unidades_consumidoras"), "bairro")
```

In [0]:
def perfilar_coluna(df: DataFrame, coluna: str, n: int = 10) -> None:
    """Exibe o perfil de uma coluna (texto, número/data ou array). Não devolve nada."""
    tipo = df.schema[coluna].dataType
    c = F.col(coluna)
    print(f"=== {coluna} ({tipo.simpleString()}) ===")

    if isinstance(tipo, T.ArrayType):
        tam = F.size(c)
        r = df.agg(
            F.count(F.lit(1)).alias("total"),
            F.sum(c.isNull().cast("int")).alias("listas_nulas"),
            F.min(F.when(c.isNotNull(), tam)).alias("tamanho_min"),
            F.max(tam).alias("tamanho_max"),
            F.sum((tam == 0).cast("int")).alias("listas_vazias"),
            F.sum(F.exists(c, lambda x: x.isNull()).cast("int")).alias("com_item_nulo"),
            F.sum((tam != F.size(F.array_distinct(c))).cast("int")).alias("com_item_repetido"),
        ).first().asDict()
        print(r)
        print("Distribuição de tamanhos:")
        _mostrar(df.groupBy(tam.alias("tamanho")).count().orderBy("tamanho"), 50)
        print(f"{n} maiores e {n} menores:")
        _mostrar(df.orderBy(tam.desc()).limit(n).unionByName(df.orderBy(tam.asc()).limit(n)), 2 * n)
        return

    if isinstance(tipo, T.StringType):
        estranho = r"[^\p{L}\p{N}\s.,;:/\-()&'ºª°#@+]"
        mojibake = r"[ÃÂ][\x{80}-\x{BF}]"
        r = df.agg(
            F.count(F.lit(1)).alias("total"),
            F.sum(c.isNull().cast("int")).alias("nulos"),
            F.countDistinct(c).alias("distintos"),
            F.min(F.length(c)).alias("tamanho_min"),
            F.max(F.length(c)).alias("tamanho_max"),
            F.sum((c != F.trim(c)).cast("int")).alias("espaco_nas_pontas"),
            F.sum(c.rlike(estranho).cast("int")).alias("caractere_estranho"),
            F.sum(c.rlike(mojibake).cast("int")).alias("encoding_quebrado"),
        ).first().asDict()
        print(r)
        print(f"{n} mais frequentes:")
        _mostrar(df.groupBy(coluna).count().orderBy(F.desc("count")), n)
        print(f"{n} primeiros e {n} últimos em ordem alfabética:")
        d = df.select(coluna).where(c.isNotNull()).distinct()
        _mostrar(d.orderBy(c.asc()).limit(n).unionByName(d.orderBy(c.desc()).limit(n)), 2 * n)
        print("Exemplos com caractere estranho:")
        _mostrar(df.select(coluna).where(c.rlike(estranho)).distinct(), n)
        return

    print("Resumo estatístico:")
    if isinstance(tipo, T.NumericType):
        _mostrar(df.select(coluna).summary(), 10)
    else:
        print(df.agg(F.count(c).alias("preenchidos"), F.sum(c.isNull().cast("int")).alias("nulos"),
                     F.min(c).alias("min"), F.max(c).alias("max")).first().asDict())
    print(f"{n} menores e {n} maiores:")
    nn = df.where(c.isNotNull())
    _mostrar(nn.orderBy(c.asc()).limit(n).unionByName(nn.orderBy(c.desc()).limit(n)), 2 * n)